In [1]:
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from typing import Dict, Any
from tavily import TavilyClient

from Chef.chef import tavily_client

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server" : {
            "transport" : "http",
            "url" : "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [3]:
from langchain_cohere import ChatCohere
travel_model = ChatCohere(model="command-r-08-2024", temperature=0)

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

travel_agent = create_agent(travel_model,
                     tools=tools,
                     checkpointer=InMemorySaver(),
                     system_prompt="You are a travel agent. Your job is to know the users needs and recommend flights according to that and No Follow UP Questions"
                     )

In [5]:

response = await travel_agent.ainvoke(
    {"messages" : [HumanMessage(content="Book me a flight from Patna to Delhi one passenger for 5 October")]},
    {"configurable" : {"thread_id" : "flight"}}
)

print(response["messages"][-1].content)


I've found a number of flights from Patna to Delhi on 26 September 2026. Here are the cheapest and shortest options:

## Cheapest
- Route: Patna → Delhi
- Times & duration: 22:10 → 23:45 (1h 35m)
- Cabin: Economy
- Price: 70 EUR
- Booking link: https://kiwi.com/u/wjr97q

## Shortest
- Route: Patna → Delhi
- Times & duration: 10:20 → 12:20 (2h)
- Cabin: Economy
- Price: 87 EUR
- Booking link: https://kiwi.com/u/b3wqcw

I recommend the shortest option, which is also the most expensive. Have a nice trip!


In [5]:
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> str:
    """Search the web for information"""

    return tavily_client.search(query)

I've found a number of flights from Patna to Delhi on 26 September 2026. Here are the cheapest and shortest options:

## Cheapest
- Route: Patna → Delhi
- Times & duration: 22:10 → 23:45 (1h 35m)
- Cabin: Economy
- Price: 70 EUR
- Booking link: https://kiwi.com/u/wjr97q

## Shortest
- Route: Patna → Delhi
- Times & duration: 10:20 → 12:20 (2h)
- Cabin: Economy
- Price: 87 EUR
- Booking link: https://kiwi.com/u/b3wqcw

I recommend the shortest option, which is also the most expensive. Have a nice trip!


In [ ]:
@tool
def travel_specialist(query: str) -> str:
    """Delegate flight, pricing, and travel search requests to the travel specialist"""

    result = travel_agent.invoke(
        {"messages" : [HumanMessage(content=query)]},
        {"configurable" : {"thread_id" : "isolated_travel_thread"}}
    )

    return result["messages"][-1].content
